In [9]:
import pandas as pd
import numpy as np
import kagglehub
from dotenv import load_dotenv
from pathlib import Path
from kagglehub import KaggleDatasetAdapter


In [10]:
load_dotenv()

True

In [15]:
cwd = Path.cwd()
project_root = cwd.parent

In [16]:
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "uciml/breast-cancer-wisconsin-data",
    "data.csv" 
)

In [17]:
type(df)

pandas.core.frame.DataFrame

In [18]:
df.shape

(569, 33)

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [20]:
df.columns

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='object')

In [21]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [22]:
df.drop(columns=["Unnamed: 32"], inplace=True)

In [26]:
dia = df["diagnosis"].value_counts()

In [33]:
dia_per = (dia/ dia.sum())*100
dia_per

diagnosis
B    62.741652
M    37.258348
Name: count, dtype: float64

In [34]:
df["diagnosis"] = df["diagnosis"].replace({"B":1, "M": -1})

/var/folders/69/dxynlnqj6413x3bgy4c63zhc0000gn/T/ipykernel_4329/795858076.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["diagnosis"] = df["diagnosis"].replace({"B":1, "M": -1})


In [35]:
df["diagnosis"].value_counts()

diagnosis
 1    357
-1    212
Name: count, dtype: int64

In [126]:
df_shuffled = df.sample(frac=1)

In [127]:
X_train = df_shuffled[[col for col in df_shuffled.columns if col not in ["id", "diagnosis"]]].iloc[:int(0.8*df_shuffled.shape[0])]
Y_train = df_shuffled["diagnosis"].iloc[:int(0.8*df_shuffled.shape[0])]
X_test = df_shuffled[[col for col in df_shuffled.columns if col not in ["id", "diagnosis"]]].iloc[int(0.8*df_shuffled.shape[0]):]
Y_test = df_shuffled["diagnosis"].iloc[int(0.8*df_shuffled.shape[0]):]
X_train = X_train.to_numpy()
Y_train = Y_train.to_numpy()
X_test = X_test.to_numpy()
Y_test = Y_test.to_numpy()

In [128]:
# Let's standardize the data
X_train = (X_train - X_train.mean(axis=0))/X_train.std(axis=0)
Y_train = Y_train.reshape((X_train.shape[0], 1))
X_test = (X_test - X_test.mean(axis=0))/X_test.std(axis=0)
Y_test = Y_test.reshape((X_test.shape[0], 1))

In [131]:
X_train.shape, Y_train.shape, X_test.shape, Y_test.shape

((455, 30), (455, 1), (114, 30), (114, 1))

In [132]:
def calculate_loss(w, b, X, Y, C=1.0):
    hinge_loss = np.maximum(0, 1 - Y * (X @ w + b)).sum()
    loss = 0.5 * (w.T @ w) + C * hinge_loss
    return loss.item()

In [133]:
# testing the calculate_loss function
w = np.zeros(shape=(X.shape[1]))
b = np.zeros(shape=(1))
w, b

(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 array([0.]))

In [134]:
calculate_loss(w=w, b=b, X=X_train, Y=Y_train)

207025.0

In [135]:
def check(w, b, X, Y):
    return Y * (X @ w + b) >= 1

In [136]:
def calculating_gradients(w, b, X, Y, C):
    result = check(w, b, X, Y).squeeze()
    dl_dw = w - C * (X[~result] * Y[~result]).sum(axis=0, keepdims=True).T / X.shape[0]
    dl_db = -C * Y[~result].sum()
    return dl_dw, dl_db

In [137]:
def update_params(w, b, X, Y, C, lr=3e-2):
    dl_dw, dl_db = calculating_gradients(w, b, X, Y, C)
    new_w = w - lr * dl_dw
    new_b = b - lr * dl_db
    return new_w, new_b

In [138]:
def train_soft_margin_svm(X, Y, C, lr=3e-2, epochs=10):
    w = np.zeros(shape=(X.shape[1], 1))
    b = np.zeros(shape=(1, 1))

    for epoch in range(epochs):
        w, b = update_params(w, b, X, Y, C, lr)
        cost = calculate_loss(w, b, X, Y, C)
        print(f"Epoch: {epoch+1}/{epochs}, Cost: {cost:.2f}")

    return w, b

In [139]:
t_w, t_b = train_soft_margin_svm(
    X=X_train,
    Y=Y_train,
    lr=1e-3,
    epochs=100,
    C=5.0
)

Epoch: 1/100, Cost: 1861.42
Epoch: 2/100, Cost: 1726.84
Epoch: 3/100, Cost: 1884.02
Epoch: 4/100, Cost: 1485.77
Epoch: 5/100, Cost: 1533.22
Epoch: 6/100, Cost: 1700.10
Epoch: 7/100, Cost: 1337.66
Epoch: 8/100, Cost: 1296.50
Epoch: 9/100, Cost: 1293.99
Epoch: 10/100, Cost: 1395.69
Epoch: 11/100, Cost: 1218.77
Epoch: 12/100, Cost: 1276.57
Epoch: 13/100, Cost: 1149.87
Epoch: 14/100, Cost: 1171.08
Epoch: 15/100, Cost: 1078.21
Epoch: 16/100, Cost: 1064.53
Epoch: 17/100, Cost: 1013.27
Epoch: 18/100, Cost: 972.38
Epoch: 19/100, Cost: 920.06
Epoch: 20/100, Cost: 904.18
Epoch: 21/100, Cost: 867.75
Epoch: 22/100, Cost: 840.88
Epoch: 23/100, Cost: 813.39
Epoch: 24/100, Cost: 774.94
Epoch: 25/100, Cost: 775.92
Epoch: 26/100, Cost: 739.22
Epoch: 27/100, Cost: 734.16
Epoch: 28/100, Cost: 695.43
Epoch: 29/100, Cost: 690.50
Epoch: 30/100, Cost: 661.81
Epoch: 31/100, Cost: 655.51
Epoch: 32/100, Cost: 634.56
Epoch: 33/100, Cost: 617.84
Epoch: 34/100, Cost: 611.45
Epoch: 35/100, Cost: 603.28
Epoch: 36/10

In [140]:
def predictions(trained_w, trained_b, X):
    raw_preds = X @ trained_w + trained_b
    raw_preds = np.where(raw_preds > 0, 1, -1)
    return raw_preds

In [141]:
def accuracy(pred, labels):
    results = np.where(pred==labels, 1, 0)
    acc = results.sum()/len(results)*100
    return acc.item()

In [142]:
training_pred = predictions(
    trained_w=t_w,
    trained_b=t_b,
    X=X_train
)

In [143]:
# checking training set accuracy
train_acc = accuracy(
    pred=training_pred,
    labels=Y_train
)
print(f"Accuracy on training data: {train_acc:.2f} %")

Accuracy on training data: 95.38 %


In [144]:
# checking test set accuracy
test_pred = predictions(
    trained_w=t_w,
    trained_b=t_b,
    X=X_test
)

In [145]:
test_acc = accuracy(
    pred=test_pred,
    labels=Y_test
)
print(f"Accuracy on test data: {test_acc:.2f} %")

Accuracy on test data: 91.23 %


In [147]:
np.unique(test_pred)

array([-1,  1])